![](https://files.seeusercontent.com/2026/09/16/k8eV/20260916104346802.png)

# 0. 总纲

- rank-1 RNN 的学习可以由少量的 scalar overlaps 描述 (loss visible $\sigma$ + loss invisible $\tilde{\sigma}$)
    - loss visible $\sigma$: 决定当前 neural dynamics, outputs 和 loss
    - loss invisible $\tilde{\sigma}$: 不改变当前 function, 但改变 gradient descent 的方向和速度

- 结论: 
    1. 两个表现完全相同的 RNN(function 相同), hidden connectivity 可以不同
    2. hidden connectivity 的差异会导致 future learning 的差异
    3. learning invariants 被解除时, invisible overlaps 可以相较于 visible overlaps 保存的 training history 被更稳健地读出

# 1. Introduction: 从 function-connectivity 的 degeneracy 出发

- Degeneracy: microscopic parameters(比如 synaptic weights) $\vec{\theta}_{1}\neq \vec{\theta}_{2}$, 但是 behavior/function $F(\vec{\theta}_{1}) = F(\vec{\theta}_{2})$
- low-rank RNN: $N$ 神经元的 connectivity 很高维($N^{2}$), 但是网络 dynamics 可以由少数的 macroscopic overlaps 表述
- Ger & Barak: overlap 这套方法, 除了描述 network dynamics, 能否描述 training dynamics?
    - 结论: 可以. 但是相比描述 activity 需要更多的 overlap (loss-invisible variables)



# 2. Section 2: Preliminaries(定义 network 和 learning)

- Definition of overlap
    - $N$ 维矢量 $\vec{u},\vec{v}\in\mathbb{R}^{N}$
    - scaled overlap: $\begin{aligned}\sigma_{vu} = \frac{1}{N}\vec{v}^{\top}\vec{u}\end{aligned}$
    - scaled squared norm: $\begin{aligned}||\vec{v}||^{2} = \frac{1}{N}\vec{v}^{\top}\vec{v}\end{aligned}$

- Time
    - $t$: within-trial time, 即神经动力学时间 (network state 的运动, state dynamics)
    - $\tau$: learning time, 跨 trial 的训练时间 (network parameters 的学习, learning dynamics)

- RNN
    - $N$: 神经元数量
    - 网络 hidden state: $\vec{h}(t) \in \mathbb{R}^{N}$
    - 动力学: $\begin{aligned}\dot{\vec{h}}(t) = -\vec{h}(t) + \mathbf{W}\phi[\vec{h}(t)] + \vec{m}x(t)\end{aligned}$
        - $\mathbf{W}\in\mathbb{R}^{N\times N}$: recurrent connectivity
        - $\phi(\cdot)$: activation function
        - $x(t)\in\mathbb{R}$: scalar input
        - $\vec{m}\in\mathbb{R}^{N}$: input direction/weights
    - 输出: $\begin{aligned}\hat{y}(t) = \frac{1}{N}\vec{z}^{\top}\phi(\vec{h}(t))\end{aligned}$
        - $\vec{z}\in\mathbb{R}^{N}$: readout vector
    - loss function: $\begin{aligned}L = \int_{0}^{T}[\hat{y}(t)-y^{\star}(t)]^{2}\mathrm{d}t\end{aligned}$
        - $y^{\star}(t)$: target output

- Learning dynamics
    - parameters: $\vec{\theta} = \{\vec{m},\mathbf{W},\vec{z}\}$
    - gradient-flow limit: $\begin{aligned}\dot{\vec{\theta}} \equiv \frac{\mathrm{d}\vec{\theta}}{\mathrm{d}\tau} = -\vec{\nabla}_{\vec{\theta}}L\end{aligned}$
        - 相当于 gradient descent 取 $\eta\to 0$ 极限
    - 问题: $\vec{h}(t)$ 可以通过 overlap 降维为 $K$ 个 $\kappa(t)$ 进行描述, 那么 $\vec{\theta}(\tau)$ 能否通过 overlap 降维为几个 scalar variables 进行描述? 

# 3. Section 3: low-rank linear RNN 推导

> linear activation: $\phi(h) = h$; rank-1 recurrent connectivity

- rank-1 connectivity: 构造 $\begin{aligned}\mathbf{W} = \frac{1}{N}\vec{u}\vec{v}^{\top}\end{aligned}$ ($u,v\in\mathbb{R}^{N}$)
    - hidden state dynamics: $\begin{aligned}\dot{\vec{h}}(t) = -\vec{h}(t) + \frac{1}{N}\vec{u}\vec{v}^{\top}\vec{h}(t) + \vec{m}x(t)\end{aligned}$
    - $\vec{u}$: recurrent interaction 的 output direction
    - $\vec{v}$: 从 activity 读取到 recurrent interaction 的 input direction
    - parameters: $\vec{\theta} = \{\vec{m}, \vec{u}, \vec{v}, \vec{z}\}$ ($4N$ 个变量)

- state dynamics: $N$ 维 $\to$ 2D
    - 初态设定 $\vec{h}(0) = \vec{0}$
    - 根据动力学 $\begin{aligned}\dot{\vec{h}} = -\vec{h} + \frac{1}{N}{\color{red}{\vec{u}}}\left(\vec{v}^{\top}\vec{h}\right) + {\color{red}{\vec{m}}}x\end{aligned}$, 因此能产生 activity 的方向是 $\vec{m}$ 和 $\vec{u}$. 因此 hidden state trajectory 只能停留在 $\mathrm{span}\{\vec{m},\vec{u}\}$
    - 分解 hidden state: $\vec{h}(t) = \kappa_{m}(t)\vec{m} + \kappa_{u}(t)\vec{u}$, 通过 $\vec{\kappa} = \begin{bmatrix} \kappa_{m} \\ \kappa_{u} \end{bmatrix}$ 重写 hidden state dynamics: 

        $$
        \begin{aligned}
        \dot{\kappa}_{m}\vec{m}+ \dot{\kappa}_{u}\vec{u} &= -\kappa_{m}\vec{m}-\kappa_{u}\vec{u} + \frac{1}{N}\vec{u}\left(\kappa_{m}\vec{v}^{\top}\vec{m} + \kappa_{u}\vec{v}^{\top}\vec{u}\right) + \vec{m}x\\
        \Rightarrow
        \dot{\kappa}_{m}\vec{m} &= (-\kappa_{m} + x)\vec{m}\\
        \dot{\kappa}_{u}\vec{u} &= \left[-\kappa_{u} + \kappa_{m}\frac{\vec{v}^{\top}\vec{m}}{N} + \kappa_{u}\frac{\vec{v}^{\top}\vec{u}}{N}\right] = (-\kappa_{u} + \kappa_{m}\sigma_{vm} + \kappa_{u}\sigma_{vu})\vec{u}\\
        \Rightarrow \dot{\vec{\kappa}} &= - \vec{\kappa} + \begin{bmatrix} 0 & 0 \\ \sigma_{vm} & \sigma_{vu} \end{bmatrix}\vec{\kappa} + \begin{bmatrix} 1 \\ 0 \end{bmatrix}x\\
        \kappa_{m}(0) = 0 \Rightarrow \kappa_{m}(t) &= \int_{0}^{t} e^{-(t-t^{\prime})}x(t^{\prime})\mathrm{d}t^{\prime}\\
        \sigma_{vu}\neq 0\Rightarrow \kappa_{u}(t) &= \sigma_{vm}\int_{0}^{t} e^{-(1-\sigma_{vu})(t-t^{\prime})}\kappa_{m}(t^{\prime})\mathrm{d}t^{\prime} = \frac{\sigma_{vm}}{\sigma_{vu}}\int_{0}^{t}\bigg[e^{-(1-\sigma_{vu})(t-t^{\prime})} - e^{-(t-t^{\prime})}\bigg]x(t^{\prime})\mathrm{d}t^{\prime}
        \end{aligned}
        $$
    - 输出: $\begin{aligned}\hat{y} = \frac{1}{N}\vec{z}^{\top}\vec{h}(t) = \frac{1}{N}\vec{z}^{\top}(\kappa_{m}\vec{m} + \kappa_{u}\vec{u}) = \frac{1}{N}\vec{z}^{\top}\vec{m}\kappa_{m} + \frac{1}{N}\vec{z}^{\top}\vec{u}\kappa_{u} = \begin{bmatrix} \sigma_{zm} & \sigma_{zu} \end{bmatrix}\vec{\kappa}\end{aligned}$
    - loss: $\begin{aligned}L = \int_{0}^{T}(\hat{y}-y^{\star})^{2}\mathrm{d}t = L(\vec{\sigma})\end{aligned}$
    - state dynamics 需要的实际上只有 overlaps $\vec{\sigma} = (\sigma_{zm}, \sigma_{zu}, \sigma_{vm}, \sigma_{vu})$. 



- learning dynamics 并不满足 $\dot{\vec{\sigma}} = -\vec{\nabla}_{\vec{\sigma}} L$, 即使 $L = L(\vec{\sigma})$
    - 真实训练: $\dot{\vec{\theta}} = -\vec{\nabla}_{\vec{\theta}}L$, overlaps 是 $\vec{\theta}$ 的函数: $\vec{\sigma} = \vec{G}(\vec{\theta})$
    - 研究 overlap map($\vec{G}$) 的 Jacobian $\begin{aligned}\mathbf{D}(\vec{\theta}) = \frac{\partial\vec{\sigma}}{\partial\vec{\theta}} = \begin{bmatrix} \frac{\partial\sigma_{1}}{\partial\theta_{1}} & \frac{\partial\sigma_{1}}{\partial\theta_{2}} & \cdots & \frac{\partial\sigma_{1}}{\partial\theta_{4N}} \\ \frac{\partial\sigma_{2}}{\partial\theta_{1}} & \frac{\partial\sigma_{2}}{\partial\theta_{2}} & \cdots & \frac{\partial\sigma_{2}}{\partial\theta_{4N}} \\ \frac{\partial\sigma_{3}}{\partial\theta_{1}} & \frac{\partial\sigma_{3}}{\partial\theta_{2}} & \cdots & \frac{\partial\sigma_{3}}{\partial\theta_{4N}} \\ \frac{\partial\sigma_{4}}{\partial\theta_{1}} & \frac{\partial\sigma_{4}}{\partial\theta_{2}} & \cdots & \frac{\partial\sigma_{4}}{\partial\theta_{4N}} \end{bmatrix}\in\mathbb{R}^{4\times 4N}\end{aligned}$
    - 链式求导: $\vec{\nabla}_{\vec{\theta}}L = \mathbf{D}^{\top}\vec{\nabla}_{\vec{\sigma}}L$

        > 证明: parameters 分量 $\theta_{i} (i=1,\cdots, 4N)$, overlaps 分量 $\sigma_{j} (j=1,2,3,4)$. loss $L = L(\sigma_{1}, \sigma_{2}, \sigma_{3}, \sigma_{4})$
        > 
        > 链式求导 $\begin{aligned}\frac{\partial L}{\partial\theta_{i}} = \sum_{j=1}^{4}\frac{\partial L}{\partial\sigma_{j}}\frac{\partial\sigma_{j}}{\partial\theta_{i}} = \sum_{j=1}^{4}D_{ji}\frac{\partial L}{\partial\sigma_{j}} = \sum_{j=1}^{4}(D^{\top})_{ij}\frac{\partial L}{\partial\sigma_{j}}\end{aligned}$
        > 
        > 写作 $\nabla$ 算子形式: $\begin{aligned}\vec{\nabla}_{\vec{\theta}}L = \mathbf{D}^{\top}\vec{\nabla}_{\vec{\sigma}}L\end{aligned}$

    - training dynamics: $\begin{aligned}\frac{\mathrm{d}\vec{\sigma}}{\mathrm{d}\tau} = \frac{\partial\vec{\sigma}}{\partial\vec{\theta}}\frac{\partial\vec{\theta}}{\partial \tau}\Rightarrow \dot{\vec{\sigma}} = \mathbf{D}\dot{\vec{\theta}} = -\mathbf{DD}^{\top}\vec{\nabla}_{\vec{\sigma}}L = -\mathbf{G}(\vec{\theta})\vec{\nabla}_{\vec{\sigma}}L\end{aligned}$
        - learning direction 是 $-\mathbf{G}(\vec{\theta})\vec{\nabla}_{\vec{\sigma}}L$ 而非 $-\vec{\nabla}_{\vec{\sigma}}L$
        - 显式计算 
        
            $$
            \begin{aligned}
            \mathbf{G}(\vec{\theta}) &= \frac{1}{N}\begin{bmatrix} 
            ||\vec{m}||^{2} + ||\vec{z}||^{2} & \sigma_{mu} & \sigma_{zv} & 0 \\ 
            \sigma_{mu} & ||\vec{u}||^{2} + ||\vec{z}||^{2} & 0 & \sigma_{zv} \\
            \sigma_{zv} & 0 & ||\vec{m}||^{2} + ||\vec{v}||^{2} & \sigma_{mu} \\
            0 & \sigma_{zv} & \sigma_{mu} & ||\vec{u}||^{2} + ||\vec{v}||^{2}
            \end{bmatrix} \\
            &= \mathbf{G} (\sigma_{mu}, \sigma_{zv}, ||\vec{m}||^{2}, ||\vec{u}||^{2}, ||\vec{v}||^{2}, ||\vec{z}||^{2}) = \mathbf{G} (\tilde{\vec{\sigma}})
            \end{aligned}$$
        
        - loss-invisible overlaps: $\tilde{\vec{\sigma}} = (\sigma_{mu}, \sigma_{zv}, ||\vec{m}||^{2}, ||\vec{u}||^{2}, ||\vec{v}||^{2}, ||\vec{z}||^{2})$
        - $(\vec{\sigma},\tilde{\vec{\sigma}})\in\mathbb{R}^{10}$ 共同构造出 training dynamics 和 state dynamics

<img src="https://files.seeusercontent.com/2026/09/16/8geD/20260916133221601.png" width="80%">

- Figure 1. 
    - (a) RNN 可通过等效的少量 overlaps 系统进行描述
    - (b) 在 $\vec{\theta}$ 空间和 $\vec{\sigma}$ 空间中 training trajectory 的映射关系($\mathbf{G}(\vec{\theta})$). 投影后的 training dynamics 并不等于在 $\vec{\sigma}$ 空间中直接进行 gradient descent
    - (c) $\vec{\theta}$ 等效于 $\vec{\sigma}$ 加 $\mathbf{G}(\vec{\theta})$, 而并不等效于单独的 $\vec{\sigma}$

# 4. Section 4: 对 dynamics 进行 loss visible/invisible decomposition 的实际物理意义

<img src="https://files.seeusercontent.com/2026/09/16/b9Fz/20260916135900463.png" width="90%">



- Figure 2. $\vec{\theta}_{1}$ 蓝色; $\vec{\theta}_{2}$ 红色
    - (a) 构造 $\vec{\sigma}(\vec{\theta}_{1}) = \vec{\sigma}(\vec{\theta}_{2})$, 但是 $\tilde{\vec{\sigma}}(\vec{\theta}_{1}) \neq \tilde{\vec{\sigma}}(\vec{\theta}_{2})$ 的两 rank-1 linear RNN
    - (b) $\vec{\sigma}(\vec{\theta}_{1}) = \vec{\sigma}(\vec{\theta}_{2})$ 相同, 而 $\tilde{\vec{\sigma}}(\vec{\theta}_{1}) \neq \tilde{\vec{\sigma}}(\vec{\theta}_{2})$ 不同
    - (c,d) hidden state dynamics, output(未开启训练) 完全相同
    - (e) $x(t)\to x(t)+\xi(t)$, output 仍然相同
    - (f) $\vec{\sigma}$ 完全相同, 而 $\tilde{\vec{\sigma}}$ 部分不同, 并且开启训练 [$\begin{cases} \dot{\vec{\sigma}}_{1} = - \mathbf{G}(\tilde{\vec{\sigma}}_{1})\vec{\nabla}_{\vec{\sigma}}L\\ \dot{\vec{\sigma}}_{2} = - \mathbf{G}(\tilde{\vec{\sigma}}_{2})\vec{\nabla}_{\vec{\sigma}}L \end{cases}$]. 结果: 输出在一开始的时候重合, 随后输出 $\hat{y}$ 出现分离
    - (g) 不同的 $\vec{\theta}$ 的 loss 降线
        - $\vec{\theta}_{1}$ 曾进入过 learning plateau, 而 $\vec{\theta}_{2}$ 没有

<img src="https://files.seeusercontent.com/2026/09/16/6Dnz/20260916204208572.png" width="90%">

- Figure 3. invisible overlaps 是否是一种 memory? 
    - (a) solution manifold
        - Task A 的 loss $L_{A}$ 只约束 visible overlaps: $L_{A} = L_{A}(\vec{\sigma})$, 而与 $\tilde{\vec{\sigma}}$ 无关. 因此 task A 的 solution 会形成 $(\vec{\sigma},\tilde{\vec{\sigma}})$ 空间中的一个 manifold $\mathcal{M}_{A} = \{(\vec{\sigma}_{A}^{\star}, \tilde{\vec{\sigma}})\}$
        - 同理, task B 也会形成解空间 $\mathcal{M}_{B} = \{(\vec{\sigma}_{B}^{\star}, \tilde{\vec{\sigma}})\}$
        - 测试训练 protocol $A_{1}\to B\to A_{2}$. Task A 只要求 $\vec{\sigma}_{A_{2}} = \vec{\sigma}_{A_{2}}$, 而并不要求 $\tilde{\vec{\sigma}}_{A_{1}} = \tilde{\vec{\sigma}}_{A_{2}}$. 因此训练后存在两种可能: 1. $(\vec{\sigma},\tilde{\vec{\sigma}})_{A_{1}} = (\vec{\sigma}, \tilde{\vec{\sigma}})_{A_{2}}$; 2. 仅 $\vec{\sigma}_{A_{1}} = \vec{\sigma}_{A_{2}}$, 而 $\tilde{\vec{\sigma}}_{A_{1}} \neq \tilde{\vec{\sigma}}_{A_{2}}$, 即通过 invisible overlaps 记录了 "曾经训练过 task B" 的信息
    - (b) linear rank-1 RNN 的训练. 使用 gradient descent ($\eta = 5\times 10^{-3}$). 
        - Task A: $y^{\star}(t) = e^{-0.2t}*x(t)$; Task B: $y^{\star}(t) = e^{-0.4t}*x(t)$
        - 对于 $A_{1}\to B_{1}\to A_{2}\to B_{2}$, 可见 $(\vec{\sigma},\tilde{\vec{\sigma}})_{A_{1}} = (\vec{\sigma},\tilde{\vec{\sigma}})_{A_{2}}$, 且 $(\vec{\sigma},\tilde{\vec{\sigma}})_{B_{1}} = (\vec{\sigma},\tilde{\vec{\sigma}})_{B_{2}}$, 即 network 连 invisible overlaps 都完全回到了原来的位置 (3a 中的情况 1)
        - $A_{3}$ 被设定为 $y^{\star}(t)\to y^{\star}(t) + \epsilon(t)$ (其中 $\epsilon(t)\sim\mathcal{N}(0,0.1^{2})$), visible overlaps $\vec{\sigma}_{A_{3}}$ 仍维持原 $\vec{\sigma}_{A_{1}} = \vec{\sigma}_{A_{2}}$ 附近, 而 invisible overlaps $\tilde{\vec{\sigma}}_{A_{3}}$ 开始逐渐偏移. 
        - 如果 visible overlaps 本身也存在 function degeneracy, noise 也可以令其产生偏移 (Figure 6)
    - (c) overlaps 的 $(A_{1},A_{2})$ 和 $(A_{1},A_{3})$ 对比
        - $(A_{1},A_{2})$: $(\vec{\sigma}_{A_{1}}, \tilde{\vec{\sigma}}_{A_{1}}) = (\vec{\sigma}_{A_{2}}, \tilde{\vec{\sigma}}_{A_{2}})$
        - $\vec{\sigma}_{A_{3}}$ 在 $\vec{\sigma}_{A_{1}}$ 附近, 而 $\tilde{\vec{\sigma}}_{A_{3}}$ 偏离 $\tilde{\vec{\sigma}}_{A_{1}}$



- conservation law: $A\to B\to A$ protocol 表现 invisible overlaps $\tilde{\vec{\sigma}}$ 的不变性
    - loss function $L(\vec{\theta}) = L(\vec{\sigma})$, 其中 $\begin{aligned}\vec{\sigma} = (\sigma_{zm}, \sigma_{zu}, \sigma_{vm}, \sigma_{vu}) = \left(\frac{1}{N}\vec{z}^{\top}\vec{m}, \frac{1}{N}\vec{z}^{\top}\vec{u}, \frac{1}{N}\vec{v}^{\top}\vec{m}, \frac{1}{N}\vec{v}^{\top}\vec{u}\right)\end{aligned}$
    - $\vec{\theta}$ 写作矩阵形式: $\begin{aligned}\mathbf{\theta} = \frac{1}{N}\begin{bmatrix}\vec{z}^{\top} \\ \vec{v}^{\top} \end{bmatrix}\begin{bmatrix}\vec{m} & \vec{u} \end{bmatrix}\end{aligned}$, 则 loss function $\begin{aligned}L(\vec{\theta}) = L\left(\frac{1}{N}\begin{bmatrix}\vec{z}^{\top} \\ \vec{v}^{\top} \end{bmatrix}\begin{bmatrix}\vec{m} & \vec{u} \end{bmatrix}\right) = L\left(\begin{bmatrix} \sigma_{zm} & \sigma_{zu} \\ \sigma_{vm} & \sigma_{vu} \end{bmatrix}\right)\end{aligned}$
        - 设 $\mathbf{A} = [\vec{z}, \vec{v}]$, $\mathbf{B} = [\vec{m},\vec{u}]$. 可见 loss 只和 $\mathbf{A}^{\top}\mathbf{B}$ 有关, 而并不分别依赖 $\mathbf{A}$ 和 $\mathbf{B}$. $\begin{aligned}\mathbf{S} = \frac{1}{N}\mathbf{A}^{\top}\mathbf{B} = \frac{1}{N}\mathbf{C} =  \begin{bmatrix} \sigma_{zm} & \sigma_{zu} \\ \sigma_{vm} & \sigma_{vu} \end{bmatrix}\end{aligned}$
        - gradient $\begin{aligned}\mathbf{J} = \frac{\partial L}{\partial (\mathbf{A}^{\top}\mathbf{B})} = \begin{bmatrix} \nabla_{zm} & \nabla_{zu} \\ \nabla_{vm} & \nabla_{vu} \end{bmatrix}\end{aligned}$
        - 微元 $\mathrm{d}\mathbf{C} = \mathrm{d}\mathbf{A}^{\top}\mathbf{B} + \mathbf{A}^{\top}\mathrm{d}\mathbf{B}$
        - loss variation $\mathrm{d}L = \mathrm{Tr}(\mathbf{J}^{\top}\mathrm{d}\mathbf{C}) = \mathrm{Tr}[\mathbf{J}^{\top}(\mathrm{d}\mathbf{A}^{\top}\mathbf{B} + \mathbf{A}^{\top}\mathrm{d}\mathbf{B})] = \mathrm{Tr}(\mathbf{J}^{\top}\mathrm{d}\mathbf{A}^{\top}\mathbf{B}) + \mathrm{Tr}(\mathbf{J}^{\top}\mathbf{A}^{\top}\mathrm{d}\mathbf{B})$
        
            利用轮换对称 $\mathrm{Tr}(ABC) = \mathrm{Tr}(BCA)$, 重新整理为 $\begin{aligned}\mathrm{d}L = \mathrm{Tr}[(\mathbf{BJ}^{\top})^{\top}\mathrm{d}\mathbf{A}] + \mathrm{Tr}[(\mathbf{AJ})^{\top}\mathrm{d}\mathbf{B}]\end{aligned}$

            而 $\begin{aligned}\mathrm{d}L = \mathrm{Tr}\left[\left(\frac{\partial L}{\partial\mathbf{A}}\right)^{\top}\mathrm{d}\mathbf{A}\right] + \mathrm{Tr}\left[\left(\frac{\partial L}{\partial\mathbf{B}}\right)^{\top}\mathrm{d}\mathbf{B}\right]\end{aligned}$, 一一对应得出 $\begin{aligned}\frac{\partial L}{\partial\mathbf{A}} = \mathbf{BJ}^{\top}, \frac{\partial L}{\partial\mathbf{B}} = \mathbf{AJ}\end{aligned}$
        - gradient descent $\dot{\vec{\theta}} = -\vec{\nabla}_{\vec{\theta}}L$, 则 $\dot{\mathbf{A}} = -\mathbf{BJ}^{\top}$, $\dot{\mathbf{B}} = -\mathbf{AJ}$
        - 注意到 $\begin{aligned}\begin{cases}\mathrm{d}_{\tau}(\mathbf{AA}^{\top}) = \dot{\mathbf{A}}\mathbf{A}^{\top} + \mathbf{A}\dot{\mathbf{A}}^{\top} = -\mathbf{BJ}^{\top}\mathbf{A}^{\top} - \mathbf{AJB}^{\top} \\ \mathrm{d}_{\tau}(\mathbf{BB}^{\top}) = \dot{\mathbf{B}}\mathbf{B}^{\top} + \mathbf{B}\dot{\mathbf{B}}^{\top} = -\mathbf{AJ}\mathbf{B}^{\top} - \mathbf{BJ}^{\top}\mathbf{A}^{\top}\end{cases}\end{aligned}$
            
            有 $\begin{aligned}\frac{\mathrm{d}}{\mathrm{d}\tau}(\mathbf{AA}^{\top}-\mathbf{BB}^{\top}) = 0\end{aligned}$, 即 $\mathbf{K} = \mathbf{AA}^{\top} - \mathbf{BB}^{\top} = \vec{z}\vec{z}^{\top} + \vec{v}\vec{v}^{\top} - \vec{m}\vec{m}^{\top} - \vec{u}\vec{u}^{\top}$ 为不变量
        - 除了 $\mathbf{K}$ 的 conservation, 还有 task 本身具有的 filter-task-specific conservation, 从而共同限制 invisible overlaps 的自由度, 确保 A-B-A protocol 的 exact recovery
    - 结论
        1. loss 虽然并不约束 invisible overlaps, 但是并不代表 learning 可以沿着 invisible overlaps direction 任意漂移 (存在 $\mathbf{K}$ constraints)
        2. 要使 $\tilde{\vec{\sigma}}$ 记录 training history, 就需要打破 $\mathbf{K}$ conservation (改变 learning rule; 改变 network architecture/dynamics...), 比如 target 加噪声, 或者 Adam 使用 adaptive optimizer

# 5. Section 5: low-rank nonlinear RNN

加入 nonlinear activation, 前面的结论还有多少是保留的? 

- 动力学: $\begin{aligned}\dot{\vec{h}}(t) = -\vec{h}(t) + \frac{1}{N}\vec{u}\vec{v}^{\top}\phi[\vec{h}(t)]  + \vec{m}x(t) = -\vec{h}(t) + {\color{red}{\vec{u}}}\left\{\frac{1}{N}\vec{v}^{\top}\phi[\vec{h}(t)]\right\}  + {\color{red}{\vec{m}}}x(t)\end{aligned}$
    - nonlinear activation function: $\begin{aligned}\phi(h) = \mathrm{erf}\left(\frac{\sqrt{\pi}}{2}h\right)\end{aligned}$ (具有性质 $\phi^{\prime}(0)=1$)
    - nonlinear activation 没有改变 latent state dimension: $\vec{h}(t)$ 仍然可以写作形式 $\vec{h}(t) = \kappa_{m}(t)\vec{m} + \kappa_{u}(t)\vec{u}$
    - $\begin{aligned}\frac{1}{N}\vec{v}^{\top}\phi(\vec{h}) = \frac{1}{N}\sum_{i}v_{i}\phi(h_{i}),\quad h_{i} = \kappa_{m}m_{i} + \kappa_{u}u_{i}\end{aligned}$, 不能简单写作 $\sigma_{vm}\kappa_{m} + \sigma_{vu}\kappa_{u}$
    - 设 Jointly Gaussian distribution: $(m_{i},u_{i},v_{i},z_{i})\sim\mathcal{N}(0,\mathbf{\Sigma})$
    
        > covariance matrix $\begin{aligned}\mathbf{\Sigma} = \begin{bmatrix} \langle m^{2}\rangle & \langle mu\rangle & \langle mv\rangle & \langle mz\rangle \\ \langle mu\rangle & \langle u^{2}\rangle & \langle uv\rangle & \langle uz\rangle \\ \langle mv\rangle & \langle uv\rangle & \langle v^{2}\rangle & \langle vz\rangle \\ \langle mz\rangle & \langle uz\rangle & \langle vz\rangle & \langle z^{2}\rangle \end{bmatrix}\end{aligned}$ 共有 10 个二阶矩作为独立变量
    - $h_{i} = \kappa_{m}(t)m_{i} + \kappa_{u}(t)u_{i}$ 也是 Gaussian. 假定 $\mathbb{E}[h_{i}(t)] = 0$, 则方差 
    
        $$\begin{aligned}
        \Delta(t) &\equiv \mathbb{E}[h_{i}(t)^{2}] = \mathbb{E}[(\kappa_{m}m_{i} + \kappa_{u}u_{i})^{2}]\\
        &= \kappa_{m}^{2}\langle m_{i}^{2}\rangle + \kappa_{u}^{2}\langle u_{i}^{2}\rangle + 2\kappa_{m}\kappa_{u}\langle m_{i}u_{i}\rangle\\
        &= \kappa_{m}^{2}||\vec{m}||^{2} + \kappa_{u}^{2}||\vec{u}||^{2} + 2\kappa_{m}\kappa_{u}\sigma_{mu}
        \end{aligned}$$

        若 $\Delta\ll 1$, 则多数 neurons 的 $h_{i}$ 都在 0 附近, 即 network 近似线性
        


- DMFT
    - $\begin{aligned}\frac{1}{N}\vec{v}^{\top}\phi(\vec{h}) = \frac{1}{N}\sum_{i}v_{i}\phi(h_{i})\end{aligned}$ 在 $N\to\infty$ 时趋近 population expectation: $\begin{aligned}\lim_{N\to\infty}\frac{1}{N}\sum_{i}v_{i}\phi(h_{i}) = \mathbb{E}[v_{i}\phi(h_{i})]\end{aligned}$
    - Stein's lemma: $\mathbb{E}[Xf(Y)] = \mathrm{Cov}(X,Y)\mathbb{E}[f^{\prime}(Y)]$, 其中 $\mathbb{E}[X] = \mathbb{E}[Y] = 0$
        - 取 $X = v_{i}$, $Y = h_{i}$, $f(\cdot) = \phi(\cdot)$
        - 计算 covariance: $\mathrm{Cov}(v_{i},h_{i}) = \mathbb{E}[v_{i}(\kappa_{m}m_{i}+\kappa_{u}u_{i})] = \kappa_{m}\sigma_{vm} + \kappa_{u}\sigma_{vu}$
        - nonlinear part: $\mathbb{E}[\phi^{\prime}(h_{i})] = \langle\phi^{\prime}\rangle$

            > $\begin{aligned}\langle\phi^{\prime}\rangle = \mathbb{E}_{h\sim\mathcal{N}(0,\Delta)}[\phi^{\prime}(h)]\end{aligned}$, 而 $\phi^{\prime}(h) = e^{-\pi h^{2}/4}$, 因此 $\begin{aligned}\mathcal{G}(\Delta) = \langle\phi^{\prime}\rangle = \int\frac{\mathrm{d}h}{\sqrt{2\pi\Delta}}e^{-h^{2}/(2\Delta)}e^{-\pi g^{2}/4} = \left(1 + \frac{\pi}{2}\Delta\right)^{-1/2}\end{aligned}$
            > 
            > $\Delta\to 0$ 时, 有展开 $\begin{aligned}\mathcal{G}(\Delta) = 1-\frac{\pi}{4}\Delta + \mathcal{O}(\Delta^2)\end{aligned}$
        - 结论: $\begin{aligned}\frac{1}{N}\vec{v}^{\top}\phi(\vec{h}) = (\kappa_{m}\sigma_{vm} + \kappa_{u}\sigma_{vu})\langle\phi^{\prime}\rangle\end{aligned}$

- 2D nonlinear effective RNN dynamics: $\dot{\vec{h}} = \dot{\kappa}_{m}\vec{m} + \dot{\kappa}_{u}\vec{u}$ 
    - 待定系数法得到 $\begin{cases} \dot{\kappa}_{m} = -\kappa_{m} + x(t) \\ \dot{\kappa}_{u} = -\kappa_{u} + (\sigma_{vm}\kappa_{m} + \sigma_{vu}\kappa_{u})\mathcal{G}(\Delta)\end{cases}$
    - readout: $\begin{aligned}\frac{1}{N}\vec{z}^{\top}\phi(\vec{h}) = \mathrm{Cov}(z_{i},h_{i})\mathcal{G}(\Delta) = (\sigma_{zm}\kappa_{m} + \sigma_{zu}\kappa_{u})\mathcal{G}(\Delta)\end{aligned}$

- overlaps 的 visible/invisible 属性变化: $4+6\to 7+3$
    - $\Delta = ||\vec{m}||^{2}\kappa_{m}^{2} + ||\vec{u}||^{2}\kappa_{u}^{2} + 2\sigma_{mu}\kappa_{m}\kappa_{u}$ 进入到 $\hat{y}$ 中而影响 loss, 因此 $\sigma_{mu}, ||\vec{m}||^{2}, ||\vec{u}||^{2}$ 变为 visible overlaps
    - nonlinear RNN
        - visible overlaps: $\vec{\sigma} = (\sigma_{zm},\sigma_{zu},\sigma_{vm}, \sigma_{vu}, {\color{red}{\sigma_{mu}, ||\vec{m}||^{2}, ||\vec{u}||^{2}}})$
        - invisible overlaps: $\tilde{\vec{\sigma}} = (\sigma_{zv}, ||\vec{z}||^{2}, ||\vec{v}||^{2})$

- training dynamics: $\dot{\vec{\sigma}} = -\mathbf{G}(\vec{\theta})\vec{\nabla}_{\vec{\sigma}}L$ 仍然成立, 只是 $\vec{\sigma}\in\mathbb{R}^{7}$

    相比于 $N$-neuron rank-1 网络, 可以使用等效的 $(\vec{\sigma},\tilde{\vec{\sigma}})\in\mathbb{R}^{10}$ 进行训练

- Figure 4. Flip-flop task 验证 nonlinear rank-1 RNN 的 visible/invisible overlaps 理论

    <img src="https://files.seeusercontent.com/2026/09/17/rRr2/20260917191048513.png" width="90%">
    
    - (a) 任务: input 中包含 $\pm 1$ 的 pulses, 需要网络记住最后一个 pulse 的符号. 因此可以推测该任务需要形成 bistability, 而 bistability 在线性系统中无法实现

        > 根据动力学 $\begin{cases} \dot{\kappa}_{m} = -\kappa_{m} + x(t) \\ \dot{\kappa}_{u} = -\kappa_{u} + (\sigma_{vm}\kappa_{m} + \sigma_{vu}\kappa_{u})\mathcal{G}(\Delta)\end{cases}$, pulse $x(t)\to\kappa_{m}\to \sigma_{vm}\kappa_{m}\to\dot{\kappa}_{u}$ 

        - 黑线: target output
        - 蓝线: $N$-dimensional nonlinear RNN
        - 红色虚线: overlaps 构造出的 effective low-dimensional RNN
    - (b) 研究 training 时间尺度($\tau$) 中的网络
        - 蓝线: $\dot{\vec{\theta}} = -\vec{\nabla}_{\vec{\theta}}L$
        - 红色虚线: $\dot{\vec{\sigma}} = -\bar{\mathbf{G}}(\vec{\sigma})\vec{\nabla}_{\vec{\sigma}}L$
        - 黑线: $\dot{\vec{\sigma}} = -\vec{\nabla}_{\vec{\sigma}}L$

        可见, 通过 overlaps 构造的 effective RNN 并不是直接在 $\vec{\sigma}$ 中进行学习, 没有和黑线重合而是和蓝线重合
    
    <img src="https://files.seeusercontent.com/2026/09/17/pyE2/20260917191152077.png" width="60%">

    - (c) 检验 $(m_{i},u_{i},v_{i},z_{i})$ 是否始终满足理论假设: jointly Gaussian distribution 且 self-consistency
        - [Example] 对于 $\vec{m} = (m_{1},m_{2},\cdots,m_{N})$, 将 $m_{i}$ 视为一次 sample, 其 histogram 相当于 empirical distribution. 观察其分布是否为 Gaussian
        
            > **Q-Q plot(quantile-quantile plot)**
            > 
            > 对于随机变量 $m\sim P$, 其 cumulative distribution function(CDF) 为 $F(x) = P(m\leq x)$. quantile function 为 $Q(p) = F^{-1}(p)$, 意思是给定概率 $p$, 求出对应的分割位置 $m$. 因此, 将 empirical distribution 的 $Q(p)$ 和 Gaussian distribution 的 $G(p)$ 拟合, 如果 $P(m)$ 为 Gaussian $X = \mu + \sigma Z$ (其中 $Z\sim\mathcal{N}(0,1)$), 则 $Q(p) = \mu + \sigma G(p)$, 即 Q-Q plot 为一条直线
        - 相关系数 $\begin{aligned}r(P(m), G(m)) = \frac{\mathrm{Cov}(P(m),G(m))}{\sqrt{\mathrm{Var}(P(m))\mathrm{Var}(G(m))}}\end{aligned}$ 非常接近 $1$

    <img src="https://files.seeusercontent.com/2026/09/17/Pps4/20260917192553800.png" width="90%">

    nonlinear RNN 的 $\tilde{\vec{\sigma}} = (\sigma_{zv},||\vec{v}||^{2}, ||\vec{z}||^{2})$ 不决定 output & loss, 却会随着 training 改变. 其是否能够记录 training history? 这里训练的是 10-overlaps effective RNN 而不是 $N$-neurons RNN

    - (d) 构造完全相同初始化的参数 $\vec{\theta}_{1}(0) = \vec{\theta}_{2}(0)$, 而分别进行不同的训练 protocol $A\to C$ 和 $B\to C$. 因为最后都训练 C 所以 visible overlaps 必定相同, 对于 invisible overlaps 呢? 
        - A: flip-flop task
        - B: stimulus integration
        
            > stimulus perid 输入 $x(t) = c + \xi(t)$, 网络需要根据 stimulus period 的输入估计 hidden mean $c$ 的数值. $c = \{\pm 2, \pm 8, \pm 16\}$, target output $\begin{aligned}y^{\star} = \frac{c}{c_{\max}}\end{aligned}$, masked MSE loss $\begin{aligned}L_{B} = \int_{0}^{T}M(t)[\hat{y}(t) - y^{\star}]^2\mathrm{d}t\end{aligned}$, decision period $M(t) = 1$, 其余为 $0$
        - C: emulate the same teacher RNN

            > 另构造 teacher RNN, 对于输入 $x(t) = \xi(t)$ 输出确定 temporal output $y^{\star}(t)$, 要求 student RNN (接受 A/B task 训练后的两 RNN)学习 $x(t) = \xi(t) \to y^{\star}(t)$ 映射
    - (e) 另训练 classifier 网络, 分别只使用 ${\color{blue}{\vec{\sigma}}}$ 和 ${\color{red}{\tilde{\vec{\sigma}}}}$ 判断 RNN 的分类 (经过 task A 还是 task B 训练)
        - 初始化: 由于还没有经过 task A/B 训练, 两种方法都无法区分 RNN 的分类(Acc=0.5, 即随机猜测)
        - after A/B: 经过 task A/B 训练后, 从 ${\color{blue}{\vec{\sigma}}}$ 和 ${\color{red}{\tilde{\vec{\sigma}}}}$ 都可以精确区分(Acc=1)
        - after C: ${\color{blue}{\vec{\sigma}}}$ 可以精确区分, 而 ${\color{red}{\tilde{\vec{\sigma}}}}$ 略低于 1
        - 与 "visible overlaps 无法分类, 而 invisible overlaps 可以分类" 的猜测不符
        
            > 作者表明这是因为有限训练时间导致并不是严格的 $\vec{\sigma}_{A\to C} = \vec{\sigma}_{B\to C}$, classifier 可以察觉两者微小的差异 (比如 $\sigma_{vu}^{A\to C} = 1.5 + 0.0001, \sigma_{vu}^{B\to C} = 1.5 - 0.0001$, 对于网络功能几乎没有影响但是 classifier 可以因此分类)
    - (f) 设定 classifier 只能以有限精度读取训练后的 overlaps (读取误差 $\epsilon\sim\mathcal{N}(0,0.1)$, 那么对于 $\sigma_{vu}^{A\to C} = 1.5 + 0.0001, \sigma_{vu}^{B\to C} = 1.5 - 0.0001$ 这样的极端案例就无法区分)
        - ${\color{blue}{\vec{\sigma}}}$ 无法再精确区分, 近似于随机猜测 Acc = 0.5
        - ${\color{red}{\tilde{\vec{\sigma}}}}$ 略低于 1, 和无读取误差的情况相比几乎没有变化
        - 结论: 在完全精确的情况下, (in)visible overlaps 都可以区分 training history, 而精度有限下只有 invisible overlaps 可以做到. invisible overlaps 是更 robust 的 training history 记录

# 6. Section 6: 作者对自己工作的界定

已有 low-rank RNN: connectivity $\to$ activity dynamics

Ger & Barak: $\begin{aligned}\frac{\mathrm{d}\vec{\sigma}}{\mathrm{d}\tau} = \frac{\partial\vec{\sigma}}{\partial\vec{\theta}}\frac{\partial\vec{\theta}}{\partial \tau}\Rightarrow \dot{\vec{\sigma}} = \mathbf{D}\dot{\vec{\theta}} = -\mathbf{DD}^{\top}\vec{\nabla}_{\vec{\sigma}}L = -\mathbf{G}(\vec{\theta})\vec{\nabla}_{\vec{\sigma}}L\end{aligned}$